In [0]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import current_timestamp, to_utc_timestamp
from pyspark.sql.functions import when, col
from pyspark.sql.functions import concat_ws
from pyspark.sql.functions import lit

In [0]:
dbutils.widgets.text("job_id", "")
job_id = dbutils.widgets.get("job_id")


In [0]:
watermark= spark.sql(f"""
SELECT watermark_column 
FROM pt.dev_metadata_schema.tbl_parameters 
where job_id='{job_id}' 
""")
watermark=watermark.collect()[0][0]

In [0]:
print(watermark)

In [0]:
fact_df = spark.sql(f"""
   SELECT * from pt.dev_bronze.fact_integcosting
   WHERE to_timestamp(upd_date)
      > to_timestamp('{watermark}')
   """)

In [0]:
fact_df=fact_df.withColumn('description', concat_ws('_', col('project_id'), col('tier_id'), lit('GE-commercial-Vernova-2026')))

In [0]:
if  fact_df.isEmpty():
    dbutils.notebook.exit("no_data")

In [0]:
fact_df.write.format("delta") \
        .option("path", "abfss://silver@projecttrackingadlsdev.dfs.core.windows.net/fact_integcosting") \
        .mode("append") \
        .saveAsTable("pt.dev_silver.fact_integcosting")

spark.sql(f"""
        update pt.dev_metadata_schema.tbl_parameters set watermark_column=current_timestamp
        where job_id=203  """)

In [0]:
dbutils.notebook.exit("loaded")